<a href="https://colab.research.google.com/github/morampudisunil/colab-files/blob/main/main_final_colab_file_for_three_models_of_conversational_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Whisper large v2**

In [ ]:
# 1) Install ffmpeg (required by Whisper)
!apt-get update -qq && apt-get install -y -qq ffmpeg

# 2) Try to overwrite problematic blinker if preinstalled
!pip install --upgrade --ignore-installed blinker

# 3) Install Python packages (quiet)
!pip install -q streamlit pyngrok transformers
# Install whisper from GitHub (more reliable)
!pip install -q git+https://github.com/openai/whisper.git

!pip install -q --upgrade "torch" "torchaudio" "torchvision" --index-url https://download.pytorch.org/whl/cpu


# 4) (Optional) If you want the faster pyTorch builds explicitly (Colab usually has torch)
# !pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu


In [27]:
!pip install transformers==4.37.2
!pip install tokenizers==0.15.2
!pip install accelerate==0.27.2
!pip install sentencepiece

In [14]:
%%writefile app.py
import os
import streamlit as st
import whisper
from transformers import pipeline
import torch, whisper
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize models
model = whisper.load_model("large-v2",device=device)
sentiment_pipeline = pipeline("sentiment-analysis")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Create upload folder if it doesn't exist
if not os.path.exists("uploads"):
    os.makedirs("uploads")

# Functions
def transcribe_audio(audio_path):
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)
    result = model.transcribe(audio)
    return result['text']

def chunk_text(text, max_tokens=500):
    sentences = text.split('. ')
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk.split()) + len(sentence.split()) <= max_tokens:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

def summarize_text(text):
    chunks = chunk_text(text)
    summaries = [
        summarizer(chunk, max_length=130, min_length=30, do_sample=False)[0]['summary_text']
        for chunk in chunks
    ]
    return " ".join(summaries)

def sentiment_analysis(text):
    return sentiment_pipeline(text)

# Streamlit UI setup
st.set_page_config(page_title="Audio Sentiment Analyzer", page_icon="🎧")
st.markdown("<h1 style='text-align: center; color: #3366cc;'>🎧 Audio Transcription & Sentiment Analysis (whisper large v2)</h1>", unsafe_allow_html=True)

uploaded_file = st.file_uploader("Upload an audio file (.mp3 or .wav)", type=["mp3", "wav"])

if uploaded_file:
    audio_path = os.path.join("uploads", uploaded_file.name)
    with open(audio_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    st.audio(audio_path, format='audio/mp3')
    st.info(f"Transcribing `{uploaded_file.name}`...")

    # Transcribe
    transcription = transcribe_audio(audio_path)
    st.subheader("📝 Transcription")
    st.success(transcription)

    # Summarize
    st.info("Summarizing the transcription...")
    summary = summarize_text(transcription)
    st.subheader("📄 Summary")
    st.info(summary)

    # Sentiment Analysis on summarized text
    sentiment = sentiment_analysis(summary)[0]
    label = sentiment["label"]
    score = sentiment["score"]

    # Color map
    color = "#ff4d4d" if label == "NEGATIVE" else "#2ecc71" if label == "POSITIVE" else "#f1c40f"

    st.subheader("🔍 Sentiment Analysis on Summary")
    st.markdown(f"""
        <div style='background-color:{color}; padding:20px; border-radius:10px;'>
            <h3 style='color:white;'>Sentiment: {label}</h3>
            <p style='color:white;'>Confidence Score: {score:.2f}</p>
        </div>
    """, unsafe_allow_html=True)

    # Optional: Clean up after processing
    # os.remove(audio_path)


Overwriting app.py


In [15]:
from pyngrok import ngrok
# REPLACE with your token string if you haven't set it elsewhere
ngrok.set_auth_token("3Af3rPCeXyAeH0BLvOjN3YcBXMK_7hCYvVvGaCbgQws2iof3Y")


In [16]:
# Start Streamlit in background and copy logs to nohup.out
!nohup streamlit run app.py >/content/nohup.out 2>&1 &


In [17]:
from pyngrok import ngrok, conf
# If you want a custom region or options, configure conf.get_default().region = "us"
tunnel = ngrok.connect(addr="8501")
print("Public URL:", tunnel.public_url)


Public URL: https://unlibellously-unthick-dorsey.ngrok-free.dev


**Whisper Medium**













In [23]:

%%writefile app.py
import os
import streamlit as st
import whisper
from transformers import pipeline

# Initialize models
model = whisper.load_model("medium")
sentiment_pipeline = pipeline("sentiment-analysis")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Create upload folder if it doesn't exist
if not os.path.exists("uploads"):
    os.makedirs("uploads")

# Functions
def transcribe_audio(audio_path):
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)
    result = model.transcribe(audio)
    return result['text']

def chunk_text(text, max_tokens=500):
    sentences = text.split('. ')
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk.split()) + len(sentence.split()) <= max_tokens:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

def summarize_text(text):
    chunks = chunk_text(text)
    summaries = [
        summarizer(chunk, max_length=130, min_length=30, do_sample=False)[0]['summary_text']
        for chunk in chunks
    ]
    return " ".join(summaries)

def sentiment_analysis(text):
    return sentiment_pipeline(text)

# Streamlit UI setup
st.set_page_config(page_title="Audio Sentiment Analyzer", page_icon="🎧")
st.markdown("<h1 style='text-align: center; color: #3366cc;'>🎧 Audio Transcription & Sentiment Analysis (Whisper Medium) </h1>", unsafe_allow_html=True)

uploaded_file = st.file_uploader("Upload an audio file (.mp3 or .wav)", type=["mp3", "wav"])

if uploaded_file:
    audio_path = os.path.join("uploads", uploaded_file.name)
    with open(audio_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    st.audio(audio_path, format='audio/mp3')
    st.info(f"Transcribing `{uploaded_file.name}`...")

    # Transcribe
    transcription = transcribe_audio(audio_path)
    st.subheader("📝 Transcription")
    st.success(transcription)

    # Summarize
    st.info("Summarizing the transcription...")
    summary = summarize_text(transcription)
    st.subheader("📄 Summary")
    st.info(summary)

    # Sentiment Analysis on summarized text
    sentiment = sentiment_analysis(summary)[0]
    label = sentiment["label"]
    score = sentiment["score"]

    # Color map
    color = "#ff4d4d" if label == "NEGATIVE" else "#2ecc71" if label == "POSITIVE" else "#f1c40f"

    st.subheader("🔍 Sentiment Analysis on Summary")
    st.markdown(f"""
        <div style='background-color:{color}; padding:20px; border-radius:10px;'>
            <h3 style='color:white;'>Sentiment: {label}</h3>
            <p style='color:white;'>Confidence Score: {score:.2f}</p>
        </div>
    """, unsafe_allow_html=True)


Overwriting app.py


In [24]:
from pyngrok import ngrok
# REPLACE with your token string if you haven't set it elsewhere
ngrok.set_auth_token("3Af3rPCeXyAeH0BLvOjN3YcBXMK_7hCYvVvGaCbgQws2iof3Y")


In [25]:
# Start Streamlit in background and copy logs to nohup.out
!nohup streamlit run app.py >/content/nohup.out 2>&1 &


In [26]:
from pyngrok import ngrok, conf
# If you want a custom region or options, configure conf.get_default().region = "us"
tunnel = ngrok.connect(addr="8501")
print("Public URL:", tunnel.public_url)


Public URL: https://unlibellously-unthick-dorsey.ngrok-free.dev


**Whisper Base Model**

In [ ]:
!pip install transformers==4.37.2
!pip install tokenizers==0.15.2
!pip install accelerate==0.27.2
!pip install sentencepiece

In [6]:
%%writefile app.py
import os
import streamlit as st
import whisper
from transformers import pipeline




model = whisper.load_model("base")
sentiment_pipeline = pipeline("sentiment-analysis")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Create upload folder if it doesn't exist
if not os.path.exists("uploads"):
    os.makedirs("uploads")

# Functions
def transcribe_audio(audio_path):
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)
    result = model.transcribe(audio)
    return result['text']

def chunk_text(text, max_tokens=500):
    sentences = text.split('. ')
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk.split()) + len(sentence.split()) <= max_tokens:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

def summarize_text(text):
    chunks = chunk_text(text)
    summaries = [
        summarizer(chunk, max_length=130, min_length=30, do_sample=False)[0]['summary_text']
        for chunk in chunks
    ]
    return " ".join(summaries)

def sentiment_analysis(text):
    return sentiment_pipeline(text)

# Streamlit UI setup
st.set_page_config(page_title="Audio Sentiment Analyzer", page_icon="🎧")
st.markdown("<h1 style='text-align: center; color: #3366cc;'>🎧 Audio Transcription & Sentiment Analysis(Whisper(base)</h1>", unsafe_allow_html=True)

uploaded_file = st.file_uploader("Upload an audio file (.mp3 or .wav)", type=["mp3", "wav"])

if uploaded_file:
    audio_path = os.path.join("uploads", uploaded_file.name)
    with open(audio_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    st.audio(audio_path, format='audio/mp3')
    st.info(f"Transcribing `{uploaded_file.name}`...")

    # Transcribe
    transcription = transcribe_audio(audio_path)
    st.subheader("📝 Transcription")
    st.success(transcription)

    # Summarize
    st.info("Summarizing the transcription...")
    summary = summarize_text(transcription)
    st.subheader("📄 Summary")
    st.info(summary)

    # Sentiment Analysis on summarized text
    sentiment = sentiment_analysis(summary)[0]
    label = sentiment["label"]
    score = sentiment["score"]

    # Color map
    color = "#ff4d4d" if label == "NEGATIVE" else "#2ecc71" if label == "POSITIVE" else "#f1c40f"

    st.subheader("🔍 Sentiment Analysis on Summary")
    st.markdown(f"""
        <div style='background-color:{color}; padding:20px; border-radius:10px;'>
            <h3 style='color:white;'>Sentiment: {label}</h3>
            <p style='color:white;'>Confidence Score: {score:.2f}</p>
        </div>
    """, unsafe_allow_html=True)


Overwriting app.py


In [7]:
from pyngrok import ngrok
# REPLACE with your token string if you haven't set it elsewhere
ngrok.set_auth_token("3Af3rPCeXyAeH0BLvOjN3YcBXMK_7hCYvVvGaCbgQws2iof3Y")


In [8]:
# Start Streamlit in background and copy logs to nohup.out
!nohup streamlit run app.py >/content/nohup.out 2>&1 &


In [9]:
from pyngrok import ngrok, conf
# If you want a custom region or options, configure conf.get_default().region = "us"
# Kill any existing ngrok tunnels to ensure a clean slate
ngrok.kill()
# Ensure that ngrok.set_auth_token("YOUR_AUTH_TOKEN") has been executed with a valid token before running this cell.
# For example, run cell 38OsJ6AMYAfK which sets the token.
tunnel = ngrok.connect(addr="8501")
print("Public URL:", tunnel.public_url)

Public URL: https://unlibellously-unthick-dorsey.ngrok-free.dev
